# Teks — SetFit: fine-tune kontrastif bi-encoder + kepala LogisticRegression (k-fold + OOF)
**Kapan dipakai:** data berlabel SEDIKIT (puluhan s.d. ribuan baris), label per kelas timpang,
atau fine-tune BERT biasa tidak stabil. SetFit (Tunstall dkk. 2022) unggul jauh di few-shot:
encoder kalimat diajari "pasangan sekelas harus mirip, beda kelas harus jauh", lalu kepala
LogisticRegression dilatih di atas embedding barunya.

**Cara:** tiap fold → bentuk pasangan (positif = kelas sama, negatif = kelas beda) dari data latih
fold → latih encoder dengan loss cosine (MSE antara cosinus dan 1/0) → embedding latih/validasi/
test → LogisticRegression. Ditulis sendiri, tanpa paket `setfit`.

**Saat lomba yang diubah:** sel Konfigurasi. Encoder kecil (`minilm`) cepat; `e5_base` lebih
kuat; `qwen3_emb_06b` paling kuat tapi lambat (pakai `POOL = "last"`, batch 8).

**Data demo:** SmSA (`sentiment`, 3 kelas), `SAMPLES_PER_CLASS = 200` meniru few-shot.

In [ ]:
import html
import os
import time

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.nn.functional import cosine_similarity, mse_loss, normalize
from transformers import AutoModel, AutoTokenizer

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/setfit_kontrastif"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"
ANSWER_CSV = f"{DATA_FOLDER}/smsa_test_labels.csv"             # HAPUS saat lomba (kunci demo)
ID_COL = "id"                    # GANTI saat lomba
TEXT_COL = "text"                # GANTI saat lomba
TARGET = "label"                 # GANTI saat lomba
SUBMIT_COL = TARGET
N_FOLDS = 5
SPLIT_SEED = 42
NOTEBOOK = "setfit_kontrastif"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"

In [ ]:
PRETRAINED = "../../../pretrained/minilm"             # GANTI: encoder kalimat multibahasa kecil
PREFIX = ""                      # awalan teks sesuai model
POOL = "mean"                    # mean (MiniLM/e5) | cls (bge-m3) | last (Qwen3-Embedding)
# PRETRAINED = "../../../pretrained/e5_base"     # lebih kuat, ± 3x lebih lambat
# PREFIX = "query: "
# PRETRAINED = "../../../pretrained/qwen3_emb_06b"   # terkuat, lambat: PAIR_BATCH 8
# PREFIX = "Instruct: Classify the text\nQuery: "
# POOL = "last"
MODEL_NAME = "setfit_minilm"     # GANTI tiap variasi (nama file OOF)
MAX_LENGTH = 128
SAMPLES_PER_CLASS = 200          # jangkar pasangan per kelas per fold. None = semua baris latih
PAIR_ROUNDS = 5                  # tiap jangkar dapat 1 pasangan positif + 1 negatif per ronde
PAIR_BATCH = 16
BODY_LR = 2e-5
HEAD_C = 1.0
SEED = 42

## Metrik

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro")


print("metrik:", METRIC_NAME, "| device:", DEVICE)

## Baca data
Teks kosong/NaN jadi string kosong, lalu `clean_basic` (html + spasi rapi). Label diubah ke indeks
0..K-1 urut `np.unique` (format OOF bersama).

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)


def clean_basic(text):
    return " ".join(html.unescape(text).split())


train_text = train[TEXT_COL].fillna("").astype(str)
test_text = test[TEXT_COL].fillna("").astype(str)
train_text = train_text.apply(clean_basic)
test_text = test_text.apply(clean_basic)
classes = np.unique(train[TARGET])
y = np.searchsorted(classes, train[TARGET].to_numpy())
print("train:", train.shape, "| test:", test.shape, "| NaN target:", train[TARGET].isna().sum())
print("kelas:", list(classes), "| jumlah:", np.bincount(y).tolist())
train[[ID_COL, TEXT_COL, TARGET]].head()

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Split (k-fold)
Aturan format OOF: `StratifiedKFold` acak seed `SPLIT_SEED` pada urutan baris train apa adanya,
jadi OOF notebook ini sefold dengan `finetune_oof`, `baseline_oof`, dll. dan bisa di-ensemble.

In [ ]:
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
folds = list(splitter.split(train_text, y))
val_sizes = []
for _, val_index in folds:
    val_sizes.append(len(val_index))
print("fold:", len(folds), "| ukuran validasi:", val_sizes)

## Pasangan kontrastif
`anchor_pool`: maksimal `SAMPLES_PER_CLASS` baris per kelas dari data latih fold. Tiap jangkar
mendapat pasangan sekelas (label 1) dan beda kelas (label 0), sebanyak `PAIR_ROUNDS` kali.

In [ ]:
all_texts = []
for text in pd.concat([train_text, test_text], ignore_index=True):
    all_texts.append(PREFIX + text)
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)
tokenizer.padding_side = "right"


def anchor_pool(fit_index, rng):
    if SAMPLES_PER_CLASS is None:
        return fit_index
    chosen = []
    for number in range(len(classes)):
        members = fit_index[y[fit_index] == number]
        size = min(SAMPLES_PER_CLASS, len(members))
        chosen.append(rng.choice(members, size=size, replace=False))
    return np.concatenate(chosen)


print("teks:", len(all_texts), "| contoh:", all_texts[0][:80])

In [ ]:
def make_pairs(anchors, fit_index, rng):
    left, right, labels = [], [], []
    for _ in range(PAIR_ROUNDS):
        for anchor in anchors:
            same = fit_index[y[fit_index] == y[anchor]]
            other = fit_index[y[fit_index] != y[anchor]]
            left.extend([anchor, anchor])
            right.extend([rng.choice(same), rng.choice(other)])
            labels.extend([1.0, 0.0])
    return np.array(left), np.array(right), np.array(labels, dtype="float32")


print("make_pairs siap | ronde:", PAIR_ROUNDS)

## Encoder
`encode` mengembalikan vektor pooling (dengan gradien saat latih). Cosine dihitung langsung.

In [ ]:
def pool(hidden, mask):
    if POOL == "cls":
        return hidden[:, 0]
    if POOL == "last":
        rows = torch.arange(hidden.shape[0], device=hidden.device)
        return hidden[rows, mask.sum(dim=1) - 1]
    weights = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1)


def encode(model, indices):
    texts = []
    for number in indices:
        texts.append(all_texts[number])
    batch = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH,
                      return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        hidden = model(**batch).last_hidden_state
    return pool(hidden.float(), batch["attention_mask"])


print("encode siap | pool:", POOL)

In [ ]:
def train_body(fit_index, rng):
    model = AutoModel.from_pretrained(PRETRAINED, local_files_only=True)
    model = model.to(DEVICE)
    model.train()
    left, right, labels = make_pairs(anchor_pool(fit_index, rng), fit_index, rng)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BODY_LR)
    order = rng.permutation(len(labels))
    for start in range(0, len(order), PAIR_BATCH):
        chosen = order[start:start + PAIR_BATCH]
        similarity = cosine_similarity(encode(model, left[chosen]), encode(model, right[chosen]))
        loss = mse_loss(similarity, torch.tensor(labels[chosen]).to(DEVICE))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"  pasangan dilatih: {len(labels)} | loss akhir {loss.item():.4f}")
    return model.eval()


print("train_body siap")

In [ ]:
def embed(model, indices):
    vectors = []
    with torch.no_grad():
        for start in range(0, len(indices), PAIR_BATCH * 4):
            part = encode(model, indices[start:start + PAIR_BATCH * 4])
            vectors.append(normalize(part, dim=1).cpu().numpy())
    return np.concatenate(vectors)


test_index = np.arange(len(train), len(train) + len(test))
print("embed siap")

## Latih k-fold
Encoder dilatih ulang tiap fold hanya dari data latih fold, jadi OOF bersih. Kepala
LogisticRegression dilatih di embedding SEMUA baris latih fold (bukan hanya jangkar).

In [ ]:
oof = np.zeros((len(train), len(classes)), dtype="float32")
test_proba = np.zeros((len(test), len(classes)), dtype="float32")
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
began = time.time()
for number, (fit_index, val_index) in enumerate(folds):
    body = train_body(fit_index, rng)
    head = LogisticRegression(C=HEAD_C, max_iter=3000)
    head.fit(embed(body, fit_index), y[fit_index])
    oof[val_index] = head.predict_proba(embed(body, val_index))
    test_proba += head.predict_proba(embed(body, test_index)) / len(folds)
    fold_value = score(y[val_index], oof[val_index].argmax(axis=1))
    print(f"fold {number}: {METRIC_NAME} {fold_value:.4f} | {time.time() - began:.0f} dtk")
    del body
    torch.cuda.empty_cache()
total_seconds = round(time.time() - began, 1)
note = (f"{os.path.basename(PRETRAINED)} | per kelas {SAMPLES_PER_CLASS} ronde {PAIR_ROUNDS} "
        f"lr {BODY_LR} len {MAX_LENGTH}")
print(note, f"| total {total_seconds / 60:.1f} menit")

## Evaluasi OOF
Skor dihitung di baris train yang punya prediksi OOF (semua baris bila semua fold jalan).

In [ ]:
filled = oof.sum(axis=1) > 0
oof_pred = oof.argmax(axis=1)
oof_score = score(y[filled], oof_pred[filled])
print(f"{METRIC_NAME} OOF ({filled.sum()} baris): {oof_score:.4f}")
print(classification_report(y[filled], oof_pred[filled], target_names=classes.astype(str)))

## Simpan OOF dan catat percobaan
`oof_<MODEL_NAME>.npy` / `test_<MODEL_NAME>.npy` hanya bila semua baris train terisi, untuk
`ensemble_oof`. `percobaan.csv` bertambah satu baris tiap run.

In [ ]:
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_proba)
    print("OOF dan test disimpan:", MODEL_NAME)
else:
    print("OOF tidak lengkap (FOLDS_TO_RUN?): tidak disimpan")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": NOTEBOOK, "model": MODEL_NAME,
       "metrik": METRIC_NAME, "skor_cv": round(oof_score, 4), "detik": total_seconds,
       "catatan": note}
log_path = f"{OOF_FOLDER}/percobaan.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(5).to_string())

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
test_pred = test_proba.argmax(axis=1)
pred_by_id = pd.Series(classes[test_pred], index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
predicted_share = submission[SUBMIT_COL].value_counts(normalize=True)
print("sebaran prediksi test:", predicted_share.round(3).to_dict())
submission.head()

## Cek submission terhadap sample_submission
Salinan validator `arsenal/umum/simpan_muat` #15 dan #16: hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas ([] = lewati cek label)
ALLOWED = list(classes)          # label sah = label train, [] = lewati cek

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
Hanya untuk data latihan yang punya kunci (`ANSWER_CSV`). Hapus sel ini saat lomba.

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth_by_id = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth_index = np.searchsorted(classes, truth_by_id.reindex(submission[ID_COL]).to_numpy())
pred_index = np.searchsorted(classes, submission[SUBMIT_COL].to_numpy())
print(f"{METRIC_NAME} di test (kunci): {score(truth_index, pred_index):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")